#### 02 - Bronze Layer: Dallas Food Inspections
**Medallion Architecture — Bronze Zone**
Raw ingestion from source CSV. No transformations. Load as-is.

- Source: City of Dallas Open Data Portal
- Target: `bronze.bronze_dallas_inspections`

##### Step 1: Configuration

In [0]:
SOURCE_PATH  = "/Volumes/workspace/default/damg7370/food_inspections/raw/Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260408.csv"
BRONZE_DB    = "bronze"
BRONZE_TABLE = "bronze_dallas_inspections"

print(f"Source : {SOURCE_PATH}")
print(f"Target : {BRONZE_DB}.{BRONZE_TABLE}")

##### Step 2: Create Bronze Database

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB}")
print(f"Database '{BRONZE_DB}' ready")

##### Step 3: Read Raw CSV

In [0]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(SOURCE_PATH)
)

print(f"Row count : {df_raw.count()}")
print(f"Columns   : {len(df_raw.columns)}")
df_raw.printSchema()

##### Step 4: Rename Columns — Remove Special Characters
Delta Lake does not allow spaces, #, or special characters in column names.

In [0]:
import re

def clean_col_name(name):
    name = name.lower().strip()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    name = re.sub(r'_+', '_', name)
    name = name.strip('_')
    return name

df_renamed = df_raw
for col_name in df_raw.columns:
    new_name = clean_col_name(col_name)
    df_renamed = df_renamed.withColumnRenamed(col_name, new_name)

print("Renamed columns:")
for c in df_renamed.columns:
    print(f"  {c}")

##### Step 5: Add Ingestion Metadata

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_bronze = (
    df_renamed
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file",         lit(SOURCE_PATH))
    .withColumn("_source_system",       lit("Dallas_OpenData"))
    .withColumn("_city",                lit("Dallas"))
)

print("Metadata columns added")
display(df_bronze.limit(3))

##### Step 6: Write to Bronze Delta Table

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_DB}.{BRONZE_TABLE}")
)

print(f"✅ Written to: {BRONZE_DB}.{BRONZE_TABLE}")

##### Step 7: Verify + Show Renamed Violation Columns

In [0]:
df_verify = spark.table(f"{BRONZE_DB}.{BRONZE_TABLE}")
row_count = df_verify.count()

print(f"✅ Row count : {row_count}")
print(f"✅ Columns   : {len(df_verify.columns)}")
assert row_count > 70000, f"Expected ~78984 rows, got {row_count}"
print("✅ Assertion passed")

viol_cols = [c for c in df_verify.columns if 'violation' in c]
print(f"\nViolation columns ({len(viol_cols)}):")
for c in viol_cols[:8]:
    print(f"  {c}")

display(df_verify.limit(5))